# DINOv3 — Chinee apple weed detection (Colab)

Self-contained Colab notebook for the `dinov3-colab` experiment in `weed-detection-experiments`.

What it does:
1. Clones the official [facebookresearch/dinov3](https://github.com/facebookresearch/dinov3) repo.
2. Loads DINOv3 weights from your Google Drive.
3. Extracts per-patch features and finds the foreground via PCA over patch tokens.
4. Launches a Gradio UI where you can drop an image and see a box around the dominant object.

**Before running:** Runtime → Change runtime type → **GPU** (T4 free is fine).

**One-time:** accept the DINOv3 license at https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/ and place the `.pth` in your Drive (default expected path is shown in step 2).

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Mount Google Drive

Weights are read from Drive so you don't re-upload them each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

Adjust paths if your Drive layout differs.

In [ ]:
import os

DINOV3_REPO = '/content/dinov3'
WEIGHTS_PATH = '/content/drive/MyDrive/dinov3/weights/dinov3_vitb16_pretrain_lvd1689m.pth'
ARCH = 'dinov3_vitb16'

assert os.path.isfile(WEIGHTS_PATH), f'Weights not found at {WEIGHTS_PATH} — update WEIGHTS_PATH or copy the .pth into Drive.'
print('Weights OK:', WEIGHTS_PATH)

## 3. Clone DINOv3 and install dependencies

In [ ]:
if not os.path.isdir(DINOV3_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/dinov3.git {DINOV3_REPO}

%pip install -q pillow scipy scikit-learn torchmetrics 'gradio>=4.0'

## 4. Inference code

Same logic as `dinov3_detect.py`, inlined so the notebook is self-contained.

In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw
from sklearn.decomposition import PCA
from scipy.ndimage import find_objects, label
from torchvision import transforms

PATCH = 16
IMG_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

model = torch.hub.load(DINOV3_REPO, ARCH, source='local', weights=WEIGHTS_PATH).to(DEVICE).eval()

_tx = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

@torch.inference_mode()
def patch_features(image):
    x = _tx(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    tokens = model.forward_features(x)['x_norm_patchtokens'][0]
    grid = IMG_SIZE // PATCH
    return tokens.float().cpu().numpy().reshape(grid, grid, -1)

def foreground_mask(features):
    h, w, d = features.shape
    proj = PCA(n_components=1).fit_transform(features.reshape(-1, d)).reshape(h, w)
    proj = (proj - proj.min()) / (proj.max() - proj.min() + 1e-8)
    if proj.mean() > 0.5:
        proj = 1.0 - proj
    return (proj > 0.5).astype(np.uint8)

def largest_component_bbox(mask):
    lbl, n = label(mask)
    if n == 0:
        return None
    sizes = np.bincount(lbl.ravel())
    sizes[0] = 0
    idx = int(sizes.argmax())
    sl = find_objects(lbl == idx)[0]
    return sl[1].start, sl[0].start, sl[1].stop, sl[0].stop

def annotate(image, bbox_grid, grid_size):
    if bbox_grid is None:
        return image
    W, H = image.size
    sx, sy = W / grid_size, H / grid_size
    x0, y0, x1, y1 = bbox_grid
    out = image.copy()
    ImageDraw.Draw(out).rectangle([x0 * sx, y0 * sy, x1 * sx, y1 * sy], outline='red', width=4)
    return out

def infer(image):
    feats = patch_features(image)
    bbox = largest_component_bbox(foreground_mask(feats))
    return annotate(image, bbox, feats.shape[0])

## 5. Launch the Gradio UI

Click the public `*.gradio.live` URL to open the interface in a new tab.

In [ ]:
import gradio as gr

gr.Interface(
    fn=infer,
    inputs=gr.Image(type='pil', label='Upload'),
    outputs=gr.Image(type='pil', label='Detected object'),
    title='DINOv3 object localization — Chinee apple',
    description='Foreground is found via PCA over DINOv3 patch tokens; the largest connected blob is boxed. Backbone-only — no class label.',
).launch(share=True)